# Smoking Detection Model Training
Train YOLOv8 to detect cigarette/smoking for Rule 13 (no-fire zone smoking detection).

Classes: `cigarette`, `smoking`, `Person`

**Runtime**: GPU (T4) - Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Step 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# Step 2: Install dependencies
!pip install -U ultralytics roboflow
from ultralytics import YOLO
print('Ready')

In [ ]:
# Step 3: Download smoking detection dataset from Roboflow
# This uses a public smoking detection dataset
# You need a free Roboflow API key: https://app.roboflow.com/

from roboflow import Roboflow

# Get your API key from: https://app.roboflow.com/settings
ROBOFLOW_API_KEY = "rf_CRArXaO4PIdGvz23b6VtvjlWEj23"  # Replace with your key

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace("yolov8-svr4x")
project = workspace.project("smoking-person-detection-ec7ec")
version = project.version(1)
dataset = version.download("yolov8")

print(f'Dataset downloaded to: {dataset.location}')
print('Dataset classes: cigarette, smoking, Person')

In [ ]:
# Step 3b: Alternative - Create a custom dataset from public images
# Use this if you don't have a Roboflow API key

import os, yaml

# Create a minimal dataset structure for smoking detection
dataset_dir = 'smoking_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{dataset_dir}/labels/{split}', exist_ok=True)

# Create dataset YAML
data_yaml = {
    'path': os.path.abspath(dataset_dir),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'Person',
        1: 'cigarette',
        2: 'smoking',
    }
}

with open(f'{dataset_dir}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print('Dataset structure created.')
print('Add your smoking images to the dataset directories.')
print('Recommended: use the Roboflow dataset (Step 3) instead.')

In [ ]:
# Step 4: Train smoking detection model

model = YOLO('yolov8n.pt')

# If using Roboflow dataset (Step 3):
data_path = 'smoking-person-detection-ec7ec-1/data.yaml'

# If using custom dataset (Step 3b):
# data_path = 'smoking_dataset/data.yaml'

results = model.train(
    data=data_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer='auto',
    lr0=0.001,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    project='runs/detect',
    name='smoking_train',
    verbose=True
)

print('\nTraining complete!')

In [ ]:
# Step 5: Validate
import glob
best_path = sorted(glob.glob('runs/detect/*/weights/best.pt'))[-1]
print(f'Model: {best_path}')

model = YOLO(best_path)
metrics = model.val(data=data_path, split='test' if os.path.exists(os.path.join(os.path.dirname(data_path), 'test')) else 'val')

print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

In [ ]:
# Step 6: Download model
from google.colab import files
import glob

best_path = sorted(glob.glob('runs/detect/*/weights/best.pt'))[-1]
files.download(best_path)
print(f'Downloaded: {best_path}')
print('\nUpload to server:')
print('  scp best.pt root@SERVER:/opt/machineVision/models/yolov8-smoking.pt')